# Experiment: App Store Trend Tracker Market Research

This notebook is the complete, standalone market research workflow for top apps across Apple App Store and Google Play.
It fetches fresh rankings, stores snapshots, finds cross-platform overlap, measures momentum, and surfaces the strongest genres.


## Objective

- Make app market research easy to run from one notebook.
- Collect current top apps for both stores.
- Track momentum against previous snapshots.
- Identify cross-platform winners and strongest genres.


In [110]:
# Setup: install any missing dependencies.
import importlib.util
import subprocess
import sys

REQUIRED = {
    "pandas": "pandas",
    "requests": "requests",
    "google-play-scraper": "google_play_scraper",
    "rapidfuzz": "rapidfuzz",
    "pyarrow": "pyarrow",
    "plotly": "plotly",
    "ipython": "IPython",
    "scikit-learn": "sklearn",
    "statsmodels": "statsmodels",
    "jinja2": "jinja2",
}

missing = [pkg for pkg, mod in REQUIRED.items() if importlib.util.find_spec(mod) is None]


def install_packages(packages: list[str]) -> None:
    pip_cmd = [sys.executable, "-m", "pip", "install", "-q", *packages]
    try:
        subprocess.check_call(pip_cmd)
        return
    except Exception as first_err:
        print(f"pip install failed: {first_err}")

    print("Trying ensurepip fallback...")
    subprocess.check_call([sys.executable, "-m", "ensurepip", "--upgrade"])
    subprocess.check_call(pip_cmd)


if missing:
    print(f"Installing missing packages: {', '.join(missing)}")
    install_packages(missing)
else:
    print("All required packages are installed.")




All required packages are installed.


In [111]:
from __future__ import annotations

from collections import defaultdict
from dataclasses import dataclass, field
from datetime import datetime, timezone
from pathlib import Path
from statistics import mean
import re
from typing import Any

import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from google_play_scraper import search

try:
    from IPython.display import display
except Exception:
    def display(obj: Any) -> None:
        print(obj)

try:
    from google_play_scraper import collection
    from google_play_scraper.constants.collection import Collection
except Exception:
    collection = None
    Collection = None

try:
    from rapidfuzz import fuzz
except Exception:
    fuzz = None

import plotly.express as px



In [112]:
@dataclass
class TrackerConfig:
    country: str = "us"
    limit: int = 50
    chart_type: str = "top-free"
    storage_dir: Path = Path("data")
    google_source_strategy: str = "auto"  # auto | chart | search
    google_queries: list[str] = field(default_factory=lambda: ["top free apps", "popular apps", "best apps"])
    timeout_sec: int = 15
    retry_count: int = 3
    retry_backoff_sec: float = 0.6
    overlap_min_score: float = 0.82

    # Strategy / modeling knobs
    new_app_days: int = 180
    strategy_top_n: int = 8
    google_detail_top_n: int = 20

    # Hidden-gem 30-day model settings
    model_horizon_days: int = 30
    hidden_gem_min_rank: int = 11
    hidden_gem_target_rank: int = 20
    hidden_gem_min_gain: float = 6.0
    max_horizon_scale: float = 2.0

    # ML tuning knobs (adjusted for hidden-gem discovery)
    model_validation_size: float = 0.2
    model_threshold_min_precision: float = 0.62


config = TrackerConfig()
config.storage_dir.mkdir(parents=True, exist_ok=True)
print(config)



TrackerConfig(country='us', limit=50, chart_type='top-free', storage_dir=PosixPath('data'), google_source_strategy='auto', google_queries=['top free apps', 'popular apps', 'best apps'], timeout_sec=15, retry_count=3, retry_backoff_sec=0.6, overlap_min_score=0.82, new_app_days=180, strategy_top_n=8, google_detail_top_n=20, model_horizon_days=30, hidden_gem_min_rank=11, hidden_gem_target_rank=20, hidden_gem_min_gain=6.0, max_horizon_scale=2.0, model_validation_size=0.2, model_threshold_min_precision=0.62)


In [113]:
SNAPSHOT_COLUMNS = [
    "captured_at_utc",
    "snapshot_id",
    "country",
    "store",
    "chart_type",
    "rank",
    "app_id",
    "app_name",
    "developer_name",
    "category",
    "source_method",
    "source_confidence",
]


def utc_now() -> datetime:
    return datetime.now(timezone.utc)


def snapshot_id(ts: datetime) -> str:
    return ts.strftime("%Y%m%dT%H%M%SZ")


def normalize_text(value: Any) -> str:
    text = str(value or "").lower().strip()
    text = re.sub(r"[^a-z0-9 ]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def similarity(a: str, b: str) -> float:
    if not a or not b:
        return 0.0
    if fuzz is not None:
        return float(fuzz.token_sort_ratio(a, b)) / 100.0
    a_set = set(a.split())
    b_set = set(b.split())
    union = len(a_set | b_set)
    return len(a_set & b_set) / union if union else 0.0


def rank_proximity(rank_a: Any, rank_b: Any) -> float:
    try:
        a = float(rank_a)
        b = float(rank_b)
    except (TypeError, ValueError):
        return 0.0
    if a <= 0 or b <= 0:
        return 0.0
    return max(0.0, 1.0 - abs(a - b) / max(a, b))


## 1. Data Ingestion (Apple + Google)

Apple uses official RSS charts. Google tries chart-style retrieval first and falls back to search-proxy ranking.


In [ ]:
def build_session(cfg: TrackerConfig) -> requests.Session:
    retry = Retry(
        total=cfg.retry_count,
        connect=cfg.retry_count,
        read=cfg.retry_count,
        status=cfg.retry_count,
        backoff_factor=cfg.retry_backoff_sec,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"],
    )
    session = requests.Session()
    adapter = HTTPAdapter(max_retries=retry)
    session.mount("https://", adapter)
    session.mount("http://", adapter)
    return session


def chunked(values: list[str], size: int) -> list[list[str]]:
    return [values[i : i + size] for i in range(0, len(values), size)]


def extract_apple_category(genres: Any) -> str:
    if isinstance(genres, list) and genres:
        first = genres[0]
        if isinstance(first, dict):
            return str(first.get("name") or "Unknown")
    return "Unknown"


def enrich_apple_categories(cfg: TrackerConfig, apple_df: pd.DataFrame) -> pd.DataFrame:
    out = apple_df.copy()
    missing_ids = (
        out.loc[out["category"].eq("Unknown") & out["app_id"].notna(), "app_id"]
        .astype(str)
        .drop_duplicates()
        .tolist()
    )
    if not missing_ids:
        return out

    session = build_session(cfg)
    lookup_map: dict[str, str] = {}
    try:
        for ids in chunked(missing_ids, 100):
            params = {
                "id": ",".join(ids),
                "country": cfg.country,
                "entity": "software",
            }
            resp = session.get("https://itunes.apple.com/lookup", params=params, timeout=cfg.timeout_sec)
            resp.raise_for_status()
            payload = resp.json()
            for item in payload.get("results", []):
                track_id = item.get("trackId")
                genre = item.get("primaryGenreName")
                if track_id is not None and genre:
                    lookup_map[str(track_id)] = str(genre)
    except Exception as exc:
        print(f"Apple category enrichment warning: {exc}")
    finally:
        session.close()

    if lookup_map:
        missing_mask = out["category"].eq("Unknown")
        out.loc[missing_mask, "category"] = (
            out.loc[missing_mask, "app_id"].astype(str).map(lookup_map).fillna("Unknown")
        )
    return out


def fetch_apple_top_charts(cfg: TrackerConfig) -> pd.DataFrame:
    url = (
        f"https://rss.applemarketingtools.com/api/v2/{cfg.country}/apps/"
        f"{cfg.chart_type}/{cfg.limit}/apps.json"
    )
    session = build_session(cfg)
    try:
        resp = session.get(url, timeout=cfg.timeout_sec)
        resp.raise_for_status()
        payload = resp.json()
        results = payload.get("feed", {}).get("results", [])
        rows = []
        for idx, app in enumerate(results, start=1):
            rows.append(
                {
                    "store": "Apple App Store",
                    "chart_type": cfg.chart_type,
                    "rank": idx,
                    "app_id": str(app.get("id", "")),
                    "app_name": str(app.get("name", "")),
                    "developer_name": str(app.get("artistName", "")),
                    "category": extract_apple_category(app.get("genres", [])),
                    "source_method": "apple_rss",
                    "source_confidence": 1.0,
                }
            )
        apple_df = pd.DataFrame(rows)
    finally:
        session.close()

    unknown_before = int((apple_df["category"] == "Unknown").sum())
    apple_df = enrich_apple_categories(cfg, apple_df)
    unknown_after = int((apple_df["category"] == "Unknown").sum())
    if unknown_after < unknown_before:
        print(f"Apple category enrichment: Unknown {unknown_before} -> {unknown_after}")

    return apple_df


def fetch_google_chart(cfg: TrackerConfig) -> pd.DataFrame:
    if collection is None or Collection is None:
        raise RuntimeError("google-play-scraper chart API unavailable in this runtime")
    raw = collection(Collection.TOP_FREE, lang="en", country=cfg.country, results=cfg.limit)
    rows = []
    for idx, app in enumerate(raw, start=1):
        rows.append(
            {
                "store": "Google Play Store",
                "chart_type": cfg.chart_type,
                "rank": idx,
                "app_id": str(app.get("appId", "")),
                "app_name": str(app.get("title", "")),
                "developer_name": str(app.get("developer", "")),
                "category": str(app.get("genre", app.get("genreId", "Unknown"))),
                "source_method": "google_chart",
                "source_confidence": 1.0,
            }
        )
    return pd.DataFrame(rows)


def fetch_google_search_proxy(cfg: TrackerConfig) -> pd.DataFrame:
    aggregate: dict[str, dict[str, Any]] = {}
    ranks_by_app: dict[str, list[int]] = defaultdict(list)

    for query in cfg.google_queries:
        raw = search(query, lang="en", country=cfg.country)
        for idx, app in enumerate(raw[: cfg.limit], start=1):
            app_id = str(app.get("appId", ""))
            if not app_id:
                continue
            aggregate.setdefault(
                app_id,
                {
                    "app_id": app_id,
                    "app_name": str(app.get("title", "")),
                    "developer_name": str(app.get("developer", "")),
                },
            )
            ranks_by_app[app_id].append(idx)

    ranked = []
    for app_id, base in aggregate.items():
        ranked.append(
            {
                **base,
                "consensus_rank": float(mean(ranks_by_app[app_id])),
                "store": "Google Play Store",
                "chart_type": cfg.chart_type,
                "category": "Mixed / Proxy",
                "source_method": "google_search_proxy",
                "source_confidence": 0.6,
            }
        )

    ranked.sort(key=lambda r: (r["consensus_rank"], r["app_name"], r["app_id"]))
    rows = []
    for idx, app in enumerate(ranked[: cfg.limit], start=1):
        rows.append(
            {
                "store": app["store"],
                "chart_type": app["chart_type"],
                "rank": idx,
                "app_id": app["app_id"],
                "app_name": app["app_name"],
                "developer_name": app["developer_name"],
                "category": app["category"],
                "source_method": app["source_method"],
                "source_confidence": app["source_confidence"],
            }
        )
    return pd.DataFrame(rows)


def fetch_google_top_apps(cfg: TrackerConfig) -> pd.DataFrame:
    strategy = cfg.google_source_strategy
    if strategy == "chart":
        return fetch_google_chart(cfg)
    if strategy == "search":
        return fetch_google_search_proxy(cfg)
    try:
        return fetch_google_chart(cfg)
    except Exception:
        return fetch_google_search_proxy(cfg)



## 2. Snapshot Management


In [ ]:
def canonicalize_snapshot(df: pd.DataFrame, cfg: TrackerConfig, captured_at: datetime, sid: str) -> pd.DataFrame:
    out = df.copy()
    out["captured_at_utc"] = captured_at.isoformat()
    out["snapshot_id"] = sid
    out["country"] = cfg.country
    for col in SNAPSHOT_COLUMNS:
        if col not in out.columns:
            out[col] = None
    out = out[SNAPSHOT_COLUMNS].copy()
    out["rank"] = pd.to_numeric(out["rank"], errors="coerce")
    out["source_confidence"] = pd.to_numeric(out["source_confidence"], errors="coerce")
    return out


def fetch_snapshot(cfg: TrackerConfig) -> pd.DataFrame:
    apple = fetch_apple_top_charts(cfg)
    google = fetch_google_top_apps(cfg)
    combined = pd.concat([apple, google], ignore_index=True)
    if combined.empty:
        raise RuntimeError("No data returned from Apple or Google")
    ts = utc_now()
    sid = snapshot_id(ts)
    return canonicalize_snapshot(combined, cfg, ts, sid)


def write_snapshot(snapshot_df: pd.DataFrame, cfg: TrackerConfig) -> dict[str, Path | None]:
    sid = str(snapshot_df["snapshot_id"].iloc[0])
    ts = pd.to_datetime(snapshot_df["captured_at_utc"].iloc[0], utc=True)
    out_dir = cfg.storage_dir / "snapshots" / f"country={cfg.country}" / f"date={ts.strftime('%Y-%m-%d')}"
    out_dir.mkdir(parents=True, exist_ok=True)

    csv_path = out_dir / f"snapshot_{sid}.csv"
    parquet_path = out_dir / f"snapshot_{sid}.parquet"
    snapshot_df.to_csv(csv_path, index=False)

    try:
        snapshot_df.to_parquet(parquet_path, index=False)
        parquet_written = parquet_path
    except Exception as exc:
        print(f"Parquet write skipped: {exc}")
        parquet_written = None

    return {"csv": csv_path, "parquet": parquet_written}


def create_snapshot(cfg: TrackerConfig) -> tuple[pd.DataFrame, dict[str, Path | None]]:
    snapshot_df = fetch_snapshot(cfg)
    paths = write_snapshot(snapshot_df, cfg)
    return snapshot_df, paths


def load_snapshot(path: Path) -> pd.DataFrame:
    if path.suffix.lower() == ".parquet":
        return pd.read_parquet(path)
    return pd.read_csv(path)


def adapt_legacy_snapshot(df: pd.DataFrame, path: Path, country: str = "us") -> pd.DataFrame:
    out = df.rename(columns={"name": "app_name", "developer": "developer_name"}).copy()
    if "captured_at_utc" not in out.columns:
        if "snapshot_date" in out.columns:
            out["captured_at_utc"] = pd.to_datetime(out["snapshot_date"], errors="coerce", utc=True).astype(str)
            out["captured_at_utc"] = out["captured_at_utc"].replace("NaT", datetime.now(timezone.utc).isoformat())
        else:
            out["captured_at_utc"] = datetime.now(timezone.utc).isoformat()
    if "snapshot_id" not in out.columns:
        out["snapshot_id"] = path.stem.replace("snapshot_", "")
    if "country" not in out.columns:
        out["country"] = country
    if "chart_type" not in out.columns:
        out["chart_type"] = "top-free"
    if "source_method" not in out.columns:
        out["source_method"] = out["store"].map(
            {
                "Apple App Store": "apple_rss",
                "Google Play Store": "google_search_proxy",
            }
        ).fillna("legacy_import")
    if "source_confidence" not in out.columns:
        out["source_confidence"] = out["source_method"].map(
            {"apple_rss": 1.0, "google_search_proxy": 0.6}
        ).fillna(0.5)
    for col in SNAPSHOT_COLUMNS:
        if col not in out.columns:
            out[col] = None
    return out[SNAPSHOT_COLUMNS].copy()


def load_snapshot_as_canonical(path: Path, cfg: TrackerConfig) -> pd.DataFrame:
    raw = load_snapshot(path)
    if set(SNAPSHOT_COLUMNS).issubset(raw.columns):
        return raw[SNAPSHOT_COLUMNS].copy()
    return adapt_legacy_snapshot(raw, path, country=cfg.country)


def list_snapshot_files(cfg: TrackerConfig) -> list[Path]:
    files: list[Path] = []
    canonical_root = cfg.storage_dir / "snapshots" / f"country={cfg.country}"
    if canonical_root.exists():
        files.extend(sorted(canonical_root.rglob("snapshot_*.parquet")))
        files.extend(sorted(canonical_root.rglob("snapshot_*.csv")))
    files.extend(sorted(cfg.storage_dir.glob("snapshot_*.csv")))

    dedup: dict[str, Path] = {}
    for path in sorted(files):
        stem = path.stem
        if stem not in dedup:
            dedup[stem] = path
            continue
        # Prefer parquet when both exist.
        if path.suffix.lower() == ".parquet":
            dedup[stem] = path
    return sorted(dedup.values())


## 3. Analysis Methods

Cross-platform matching is score-based (name + developer + rank proximity).
Momentum compares latest snapshot against previous snapshot.


In [ ]:
def confidence_tier(score: float) -> str:
    if score >= 0.90:
        return "high"
    if score >= 0.82:
        return "medium"
    return "low"


def match_cross_platform(df: pd.DataFrame, min_score: float = 0.82) -> pd.DataFrame:
    apple = df[df["store"] == "Apple App Store"].copy()
    google = df[df["store"] == "Google Play Store"].copy()

    if apple.empty or google.empty:
        return pd.DataFrame()

    apple["name_norm"] = apple["app_name"].map(normalize_text)
    apple["dev_norm"] = apple["developer_name"].map(normalize_text)
    google["name_norm"] = google["app_name"].map(normalize_text)
    google["dev_norm"] = google["developer_name"].map(normalize_text)

    candidates = []
    for a in apple.itertuples(index=False):
        for g in google.itertuples(index=False):
            name_sim = similarity(a.name_norm, g.name_norm)
            dev_sim = similarity(a.dev_norm, g.dev_norm)
            rank_sim = rank_proximity(a.rank, g.rank)
            score = 0.65 * name_sim + 0.25 * dev_sim + 0.10 * rank_sim
            if score < min_score:
                continue
            candidates.append(
                {
                    "apple_app_id": a.app_id,
                    "google_app_id": g.app_id,
                    "apple_name": a.app_name,
                    "google_name": g.app_name,
                    "apple_developer": a.developer_name,
                    "google_developer": g.developer_name,
                    "apple_rank": a.rank,
                    "google_rank": g.rank,
                    "match_score": round(score, 4),
                    "confidence": confidence_tier(score),
                }
            )

    if not candidates:
        return pd.DataFrame()

    candidates.sort(key=lambda row: row["match_score"], reverse=True)
    used_apple = set()
    used_google = set()
    selected = []
    for row in candidates:
        if row["apple_app_id"] in used_apple or row["google_app_id"] in used_google:
            continue
        used_apple.add(row["apple_app_id"])
        used_google.add(row["google_app_id"])
        selected.append(row)

    return pd.DataFrame(selected).sort_values(
        by=["match_score", "apple_rank"], ascending=[False, True]
    ).reset_index(drop=True)


def compute_momentum(current_df: pd.DataFrame, previous_df: pd.DataFrame) -> pd.DataFrame:
    keys = ["app_id", "store"]

    current = current_df[
        ["app_id", "store", "app_name", "developer_name", "category", "rank", "source_method"]
    ].copy()
    previous = previous_df[
        ["app_id", "store", "app_name", "developer_name", "category", "rank"]
    ].rename(columns={"rank": "previous_rank"}).copy()

    merged = current.merge(previous[["app_id", "store", "previous_rank"]], on=keys, how="left")
    movement = merged["previous_rank"] - merged["rank"]
    merged["status"] = "new"
    has_prev = merged["previous_rank"].notna()
    merged.loc[has_prev & (movement > 0), "status"] = "up"
    merged.loc[has_prev & (movement < 0), "status"] = "down"
    merged.loc[has_prev & (movement == 0), "status"] = "stable"
    merged["movement"] = movement

    previous_only = previous.merge(current[keys], on=keys, how="left", indicator=True)
    previous_only = previous_only[previous_only["_merge"] == "left_only"].drop(columns=["_merge"])
    if not previous_only.empty:
        previous_only = previous_only.rename(columns={"previous_rank": "rank"})
        previous_only["previous_rank"] = previous_only["rank"]
        previous_only["rank"] = pd.NA
        previous_only["source_method"] = "historical_only"
        previous_only["status"] = "dropped"
        previous_only["movement"] = pd.NA
        previous_only = previous_only[
            [
                "app_id",
                "store",
                "app_name",
                "developer_name",
                "category",
                "rank",
                "source_method",
                "previous_rank",
                "status",
                "movement",
            ]
        ]

    merged = merged[
        [
            "app_id",
            "store",
            "app_name",
            "developer_name",
            "category",
            "rank",
            "source_method",
            "previous_rank",
            "status",
            "movement",
        ]
    ]

    if not previous_only.empty:
        merged = pd.concat([merged, previous_only], ignore_index=True)

    status_order = {"up": 0, "stable": 1, "down": 2, "new": 3, "dropped": 4}
    merged["_status_order"] = merged["status"].map(status_order).fillna(99)
    merged = merged.sort_values(
        by=["_status_order", "movement", "rank", "previous_rank"],
        ascending=[True, False, True, True],
    ).drop(columns=["_status_order"])

    return merged.reset_index(drop=True)


In [ ]:
def build_reports(
    snapshot_df: pd.DataFrame,
    overlap_df: pd.DataFrame,
    momentum_df: pd.DataFrame | None,
    output_dir: Path,
) -> dict[str, Path]:
    output_dir.mkdir(parents=True, exist_ok=True)
    artifacts = {}

    store_dist = snapshot_df["store"].value_counts().rename_axis("store").reset_index(name="count")
    store_dist_path = output_dir / "store_distribution.csv"
    store_dist.to_csv(store_dist_path, index=False)
    artifacts["store_distribution"] = store_dist_path

    overlap_path = output_dir / "overlap_summary.csv"
    overlap_df.to_csv(overlap_path, index=False)
    artifacts["overlap_summary"] = overlap_path

    if momentum_df is not None and not momentum_df.empty:
        momentum_path = output_dir / "momentum_summary.csv"
        momentum_df.to_csv(momentum_path, index=False)
        artifacts["momentum_summary"] = momentum_path

    return artifacts


## 4. Run Snapshot


In [ ]:
# Strategy settings + metadata enrichment (release age, ratings, installs)
config.new_app_days = getattr(config, "new_app_days", 180)
config.google_detail_top_n = getattr(config, "google_detail_top_n", 20)
config.strategy_top_n = getattr(config, "strategy_top_n", 8)

META_COLS = [
    "release_date",
    "age_days",
    "rating_score",
    "rating_count",
    "review_count",
    "min_installs",
    "price_usd",
    "is_free",
]


def _chunk(values: list[str], size: int) -> list[list[str]]:
    return [values[i : i + size] for i in range(0, len(values), size)]


def _parse_installs(value: Any) -> float | None:
    if value is None:
        return None
    if isinstance(value, (int, float)):
        return float(value)
    txt = str(value)
    digits = re.sub(r"[^0-9]", "", txt)
    return float(digits) if digits else None


def _parse_price(value: Any) -> float:
    if value is None:
        return 0.0
    if isinstance(value, (int, float)):
        return float(value)
    txt = str(value).replace("$", "").replace(",", "")
    m = re.search(r"[0-9]+(?:\.[0-9]+)?", txt)
    return float(m.group(0)) if m else 0.0


def _ensure_meta_cols(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for col in META_COLS:
        if col not in out.columns:
            out[col] = None
    return out


def enrich_snapshot_metadata(cfg: TrackerConfig, df: pd.DataFrame) -> pd.DataFrame:
    out = _ensure_meta_cols(df)

    # Apple enrichment via iTunes lookup (batch).
    apple_ids = out.loc[out["store"].eq("Apple App Store"), "app_id"].astype(str).drop_duplicates().tolist()
    if apple_ids:
        lookup_map: dict[str, dict[str, Any]] = {}
        with requests.Session() as session:
            for ids in _chunk(apple_ids, 100):
                try:
                    resp = session.get(
                        "https://itunes.apple.com/lookup",
                        params={"id": ",".join(ids), "country": cfg.country, "entity": "software"},
                        timeout=cfg.timeout_sec,
                    )
                    resp.raise_for_status()
                    payload = resp.json()
                except Exception:
                    continue
                for item in payload.get("results", []):
                    tid = item.get("trackId")
                    if tid is None:
                        continue
                    price = _parse_price(item.get("trackPrice"))
                    lookup_map[str(tid)] = {
                        "category": item.get("primaryGenreName"),
                        "release_date": item.get("releaseDate"),
                        "rating_score": item.get("averageUserRating"),
                        "rating_count": item.get("userRatingCount"),
                        "review_count": item.get("userRatingCount"),
                        "price_usd": price,
                        "is_free": bool(price == 0 or "free" in str(item.get("formattedPrice", "")).lower()),
                    }

        for idx, row in out[out["store"].eq("Apple App Store")].iterrows():
            meta = lookup_map.get(str(row["app_id"]))
            if not meta:
                continue
            if row.get("category") in [None, "", "Unknown"] and meta.get("category"):
                out.at[idx, "category"] = str(meta["category"])
            for col in ["release_date", "rating_score", "rating_count", "review_count", "price_usd", "is_free"]:
                out.at[idx, col] = meta.get(col)

    # Google enrichment via app details for top N apps.
    try:
        from google_play_scraper import app as gp_app
    except Exception:
        gp_app = None

    if gp_app is not None:
        g_ids = (
            out[out["store"].eq("Google Play Store")]
            .sort_values("rank")["app_id"]
            .astype(str)
            .drop_duplicates()
            .head(int(cfg.google_detail_top_n))
            .tolist()
        )
        detail_map: dict[str, dict[str, Any]] = {}
        for app_id in g_ids:
            try:
                d = gp_app(app_id, lang="en", country=cfg.country)
            except Exception:
                continue
            min_installs = d.get("minInstalls")
            if min_installs is None:
                min_installs = _parse_installs(d.get("installs"))
            detail_map[app_id] = {
                "category": d.get("genre"),
                "release_date": d.get("released"),
                "rating_score": d.get("score"),
                "rating_count": d.get("ratings"),
                "review_count": d.get("reviews"),
                "min_installs": min_installs,
                "price_usd": _parse_price(d.get("price", 0)),
                "is_free": bool(d.get("free", True)),
            }

        for idx, row in out[out["store"].eq("Google Play Store")].iterrows():
            meta = detail_map.get(str(row["app_id"]))
            if not meta:
                continue
            if row.get("category") in [None, "", "Unknown", "Mixed / Proxy"] and meta.get("category"):
                out.at[idx, "category"] = str(meta["category"])
            for col in ["release_date", "rating_score", "rating_count", "review_count", "min_installs", "price_usd", "is_free"]:
                out.at[idx, col] = meta.get(col)

    rel = pd.to_datetime(out["release_date"], errors="coerce", utc=True)
    cap = pd.to_datetime(out["captured_at_utc"], errors="coerce", utc=True)
    out["age_days"] = (cap - rel).dt.days
    out["release_date"] = rel.dt.strftime("%Y-%m-%d")

    for c in ["rating_score", "rating_count", "review_count", "min_installs", "price_usd", "age_days"]:
        out[c] = pd.to_numeric(out[c], errors="coerce")

    unknown_before = int((df["category"] == "Unknown").sum()) if "category" in df.columns else -1
    unknown_after = int((out["category"] == "Unknown").sum())
    if unknown_before >= 0:
        print(f"Category enrichment: Unknown {unknown_before} -> {unknown_after}")

    return out



In [ ]:
snapshot_df, snapshot_paths = create_snapshot(config)
snapshot_df = enrich_snapshot_metadata(config, snapshot_df)

print(f"rows={len(snapshot_df)}")
print(f"csv={snapshot_paths['csv']}")
print(f"parquet={snapshot_paths['parquet']}")

display(snapshot_df.head())



Apple category enrichment: Unknown 10 -> 0
Category enrichment: Unknown 0 -> 0
rows=100
csv=data/snapshots/country=us/date=2026-03-02/snapshot_20260302T133414Z.csv
parquet=data/snapshots/country=us/date=2026-03-02/snapshot_20260302T133414Z.parquet


,captured_at_utc,snapshot_id,country,store,chart_type,rank,app_id,app_name,developer_name,category,source_method,source_confidence,release_date,age_days,rating_score,rating_count,review_count,min_installs,price_usd,is_free
0,2026-03-02T13:34:14.564613+00:00,20260302T133414Z,us,Apple App Store,top-free,1,6473753684,Claude by Anthropic,Anthropic PBC,Productivity,apple_rss,1.0,2024-05-01,670.0,4.70544,57540.0,57540.0,NaN,0.0,True
1,2026-03-02T13:34:14.564613+00:00,20260302T133414Z,us,Apple App Store,top-free,2,6448311069,ChatGPT,"OpenAI OpCo, LLC",Productivity,apple_rss,1.0,2023-05-18,1019.0,4.85113,5921294.0,5921294.0,NaN,0.0,True
2,2026-03-02T13:34:14.564613+00:00,20260302T133414Z,us,Apple App Store,top-free,3,6477489729,Google Gemini,Google,Productivity,apple_rss,1.0,2024-11-13,474.0,4.72211,1341905.0,1341905.0,NaN,0.0,True
3,2026-03-02T13:34:14.564613+00:00,20260302T133414Z,us,Apple App Store,top-free,4,1673567402,Freecash - Get Paid Real Money,256 REWARDS LTD,Entertainment,apple_rss,1.0,2024-04-01,700.0,4.77362,48887.0,48887.0,NaN,0.0,True
4,2026-03-02T13:34:14.564613+00:00,20260302T133414Z,us,Apple App Store,top-free,5,6446901002,Threads,"Instagram, Inc.",Social Networking,apple_rss,1.0,2023-07-05,971.0,4.63740,1786153.0,1786153.0,NaN,0.0,True


In [ ]:
# Store mix and Apple category breakdown.
store_distribution = snapshot_df["store"].value_counts().rename_axis("store").reset_index(name="count")
display(store_distribution)

apple_df = snapshot_df[snapshot_df["store"] == "Apple App Store"].copy()
apple_categories = apple_df["category"].value_counts().rename_axis("category").reset_index(name="count")
display(apple_categories.head(15))

fig = px.bar(
    apple_categories.head(15),
    x="category",
    y="count",
    title="Top Apple Categories in Current Snapshot",
    color="count",
    color_continuous_scale="Blues",
)
fig.show()


,store,count
0,Apple App Store,50
1,Google Play Store,50


,category,count
0,Entertainment,11
1,Shopping,8
2,Social Networking,6
3,Productivity,5
4,Photo & Video,5
5,Finance,3
6,Utilities,2
7,Travel,2
8,Food & Drink,2
9,News,1


In [ ]:
# Cross-platform overlap analysis.
overlap_df = match_cross_platform(snapshot_df, min_score=config.overlap_min_score)
print(f"Cross-platform matches: {len(overlap_df)}")
if not overlap_df.empty:
    display(overlap_df.head(20))


Cross-platform matches: 11


,apple_app_id,google_app_id,apple_name,google_name,apple_developer,google_developer,apple_rank,google_rank,match_score,confidence
0,897446215,com.canva.editor,Canva: AI Photo & Video Editor,Canva: AI Photo & Video Editor,Canva,Canva,45,48,0.9938,high
1,284882215,com.facebook.katana,Facebook,Facebook,"Meta Platforms, Inc.","Meta Platforms, Inc.",25,23,0.9920,high
2,447188370,com.snapchat.android,Snapchat,Snapchat,"Snap, Inc.",Snap Inc,50,39,0.9780,high
3,310633997,com.whatsapp,WhatsApp Messenger,WhatsApp Messenger,WhatsApp Inc.,WhatsApp LLC,16,21,0.9345,high
4,711923939,com.squareup.cash,Cash App,Cash App,"Block, Inc.","Block, Inc.",36,9,0.9250,high
5,333903271,com.twitter.android,X,X,X Corp.,X Corp.,10,43,0.9233,high
6,324684580,com.spotify.music,Spotify: Music and Podcasts,Spotify: Music and Podcasts,Spotify,Spotify AB,14,29,0.9042,high
7,835599320,com.zhiliaoapp.musically,"TikTok - Videos, Shop & LIVE","TikTok - Videos, Shop & LIVE",TikTok Ltd.,TikTok Pte. Ltd.,19,8,0.9004,high
8,284815942,com.google.android.googlequicksearchbox,Google,Google,Google,Google LLC,9,18,0.8875,medium
9,389801252,com.instagram.android,Instagram,Instagram,"Instagram, Inc.",Instagram,21,4,0.8736,medium


In [ ]:
# Momentum analysis from the latest two snapshots.
momentum_df = pd.DataFrame()
files = list_snapshot_files(config)
print(f"Snapshot files found: {len(files)}")

if len(files) >= 2:
    previous_df = load_snapshot_as_canonical(files[-2], config)
    current_df = load_snapshot_as_canonical(files[-1], config)
    momentum_df = compute_momentum(current_df, previous_df)
    display(momentum_df["status"].value_counts().rename_axis("status").reset_index(name="count"))
    display(momentum_df.head(20))
else:
    print("Need at least 2 snapshots for momentum analysis. Run this notebook again later.")


Snapshot files found: 6


/tmp/ipython-input-335/2588855943.py:123: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.



,status,count
0,stable,58
1,up,18
2,down,18
3,new,6
4,dropped,6


,app_id,store,app_name,developer_name,category,rank,source_method,previous_rank,status,movement
0,com.squareup.cash,Google Play Store,Cash App,"Block, Inc.",Mixed / Proxy,9,google_search_proxy,31.0,up,22.0
1,tv.pluto.android,Google Play Store,PlutoTV: Stream Free Movies/TV,"Pluto, Inc.",Mixed / Proxy,14,google_search_proxy,34.0,up,20.0
2,com.instagram.android,Google Play Store,Instagram,Instagram,Mixed / Proxy,4,google_search_proxy,18.0,up,14.0
3,com.pinger.textfree,Google Play Store,Text Free: Second Phone Number,"Pinger, Inc",Mixed / Proxy,7,google_search_proxy,19.0,up,12.0
4,com.snapchat.android,Google Play Store,Snapchat,Snap Inc,Mixed / Proxy,39,google_search_proxy,49.0,up,10.0
5,com.google.android.googlequicksearchbox,Google Play Store,Google,Google LLC,Mixed / Proxy,18,google_search_proxy,25.0,up,7.0
6,com.google.android.gm,Google Play Store,Gmail,Google LLC,Mixed / Proxy,26,google_search_proxy,33.0,up,7.0
7,com.philo.philo.google,Google Play Store,"Philo: Shows, Movies, Live TV.","Philo, Inc.",Mixed / Proxy,20,google_search_proxy,26.0,up,6.0
8,com.facebook.orca,Google Play Store,Messenger,"Meta Platforms, Inc.",Mixed / Proxy,34,google_search_proxy,38.0,up,4.0
9,com.fitbit.FitbitMobile,Google Play Store,Fitbit,Google LLC,Mixed / Proxy,41,google_search_proxy,45.0,up,4.0


## 5. Predictive Modeling (ML + Statistical)

This section trains models on historical snapshot transitions to forecast near-term app movement.
- ML model: predicts `will_improve` in the next snapshot.
- Statistical model: interpretable logistic regression drivers.



In [ ]:
# ML/stat modeling helpers (30-day hidden-gem objective)
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    fbeta_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
import statsmodels.api as sm


def _snapshot_timestamp(df: pd.DataFrame) -> pd.Timestamp:
    if "captured_at_utc" in df.columns and not df.empty:
        ts = pd.to_datetime(df["captured_at_utc"].iloc[0], errors="coerce", utc=True)
        if ts is not pd.NaT:
            return ts
    return pd.Timestamp.now(tz="UTC")


def _engineer_transition_features(df: pd.DataFrame, cfg: TrackerConfig) -> pd.DataFrame:
    out = df.copy()
    out["rank"] = pd.to_numeric(out["rank"], errors="coerce")
    out["source_confidence"] = pd.to_numeric(out["source_confidence"], errors="coerce").fillna(0.5)

    out["rank_inv"] = (cfg.limit + 1) - out["rank"]
    out["rank_pct"] = out["rank"] / max(cfg.limit, 1)
    out["rank_sq"] = out["rank"] ** 2
    out["is_mid_rank"] = ((out["rank"] >= cfg.hidden_gem_min_rank) & (out["rank"] <= 30)).astype(int)
    out["is_deep_rank"] = (out["rank"] > 30).astype(int)

    out["has_rating_data"] = (pd.to_numeric(out["rating_count"], errors="coerce").fillna(0) > 0).astype(int)
    out["has_install_data"] = (pd.to_numeric(out["min_installs"], errors="coerce").fillna(0) > 0).astype(int)

    out["score_density"] = out["source_confidence"] * out["rank_inv"].clip(lower=0)
    return out


def build_transition_dataset(cfg: TrackerConfig) -> pd.DataFrame:
    files = list_snapshot_files(cfg)
    if len(files) < 2:
        return pd.DataFrame()

    snapshots: list[dict[str, Any]] = []
    for path in files:
        df = load_snapshot_as_canonical(path, cfg)
        if df.empty:
            continue
        snapshots.append({"ts": _snapshot_timestamp(df), "df": df, "path": path})

    snapshots.sort(key=lambda x: x["ts"])
    if len(snapshots) < 2:
        return pd.DataFrame()

    frames = []
    for i in range(len(snapshots) - 1):
        start = snapshots[i]
        start_ts = start["ts"]

        j = None
        for k in range(i + 1, len(snapshots)):
            delta_days = (snapshots[k]["ts"] - start_ts).total_seconds() / 86400.0
            if delta_days >= cfg.model_horizon_days:
                j = k
                break
        if j is None:
            j = len(snapshots) - 1
            if j == i:
                continue

        future = snapshots[j]
        delta_days = (future["ts"] - start_ts).total_seconds() / 86400.0
        if delta_days <= 0:
            continue

        prev = start["df"].copy()
        nxt = future["df"].copy()

        nxt_rank = nxt[["app_id", "store", "rank"]].rename(columns={"rank": "next_rank"})
        merged = prev.merge(nxt_rank, on=["app_id", "store"], how="left")

        for col in [
            "is_free", "age_days", "rating_score", "rating_count", "min_installs",
            "source_confidence", "category",
        ]:
            if col not in merged.columns:
                merged[col] = np.nan

        merged["rank"] = pd.to_numeric(merged["rank"], errors="coerce")
        merged["next_rank"] = pd.to_numeric(merged["next_rank"], errors="coerce")

        # Category density within each store snapshot.
        cat_counts = merged.groupby(["store", "category"]).size().rename("category_count_prev").reset_index()
        store_counts = merged.groupby(["store"]).size().rename("store_count_prev").reset_index()
        merged = merged.merge(cat_counts, on=["store", "category"], how="left")
        merged = merged.merge(store_counts, on=["store"], how="left")
        merged["category_share_prev"] = merged["category_count_prev"] / merged["store_count_prev"].replace(0, np.nan)

        merged["rank_delta"] = merged["rank"] - merged["next_rank"]
        scale = min(cfg.model_horizon_days / max(delta_days, 1e-6), cfg.max_horizon_scale)
        merged["rank_delta_30d"] = merged["rank_delta"] * scale
        merged["projected_rank_30d"] = merged["rank"] - merged["rank_delta_30d"]

        merged["will_survive_30d"] = merged["next_rank"].notna().astype(int)
        merged["hidden_gem_30d"] = (
            (merged["rank"] >= cfg.hidden_gem_min_rank)
            & (merged["rank"] <= cfg.limit)
            & (merged["will_survive_30d"] == 1)
            & (merged["rank_delta_30d"] >= cfg.hidden_gem_min_gain)
            & (merged["projected_rank_30d"] <= cfg.hidden_gem_target_rank)
        ).astype(int)

        merged["transition_idx"] = i
        merged["delta_days"] = float(delta_days)
        merged["horizon_scale"] = float(scale)
        merged["is_google"] = (merged["store"] == "Google Play Store").astype(int)

        merged["age_days"] = pd.to_numeric(merged.get("age_days"), errors="coerce")
        merged["rating_score"] = pd.to_numeric(merged.get("rating_score"), errors="coerce")
        merged["rating_count"] = pd.to_numeric(merged.get("rating_count"), errors="coerce")
        merged["min_installs"] = pd.to_numeric(merged.get("min_installs"), errors="coerce")
        merged["source_confidence"] = pd.to_numeric(merged.get("source_confidence"), errors="coerce").fillna(0.5)
        merged["is_free"] = merged.get("is_free").fillna(False).astype(int)
        merged["category_share_prev"] = pd.to_numeric(merged.get("category_share_prev"), errors="coerce").fillna(0)

        merged["log_rating_count"] = np.log1p(merged["rating_count"].fillna(0))
        merged["log_min_installs"] = np.log1p(merged["min_installs"].fillna(0))
        merged["log_age_days"] = np.log1p(merged["age_days"].clip(lower=0).fillna(0))

        merged = _engineer_transition_features(merged, cfg)

        horizon_quality = np.minimum(merged["delta_days"], cfg.model_horizon_days) / cfg.model_horizon_days
        merged["sample_weight"] = (0.7 * horizon_quality + 0.3 * merged["source_confidence"]).clip(lower=0.1)

        frames.append(merged)

    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)


def _best_threshold(y_true: np.ndarray, proba: np.ndarray, min_precision: float = 0.55) -> float:
    best_t = 0.5
    best_s = -1.0
    fallback_t = 0.5
    fallback_s = -1.0

    for t in np.linspace(0.10, 0.90, 81):
        pred = (proba >= t).astype(int)
        p = precision_score(y_true, pred, zero_division=0)
        r = recall_score(y_true, pred, zero_division=0)
        f1 = f1_score(y_true, pred, zero_division=0)
        f05 = fbeta_score(y_true, pred, beta=0.5, zero_division=0)
        score = 0.7 * f1 + 0.3 * f05

        if score > fallback_s:
            fallback_s = score
            fallback_t = float(t)

        if p >= min_precision and score > best_s:
            best_s = score
            best_t = float(t)

    return best_t if best_s >= 0 else fallback_t


def _build_pipeline(params: dict[str, Any], num_cols: list[str], cat_cols: list[str]) -> Pipeline:
    pre = ColumnTransformer(
        transformers=[
            ("num", Pipeline([("impute", SimpleImputer(strategy="median"))]), num_cols),
            ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")), ("ohe", OneHotEncoder(handle_unknown="ignore"))]), cat_cols),
        ],
        remainder="drop",
    )

    clf = RandomForestClassifier(
        random_state=42,
        class_weight="balanced_subsample",
        n_jobs=-1,
        n_estimators=int(params.get("n_estimators", 600)),
        max_depth=params.get("max_depth", None),
        min_samples_leaf=int(params.get("min_samples_leaf", 1)),
        min_samples_split=int(params.get("min_samples_split", 2)),
        max_features=params.get("max_features", "sqrt"),
    )

    return Pipeline([("pre", pre), ("clf", clf)])


def train_ml_forecast_model(transitions: pd.DataFrame, cfg: TrackerConfig):
    target = "hidden_gem_30d"
    if transitions.empty or transitions[target].nunique() < 2:
        return None, {}, 0.5, pd.DataFrame(), pd.DataFrame()

    feature_cols = [
        "rank",
        "rank_inv",
        "rank_pct",
        "rank_sq",
        "store",
        "category",
        "source_confidence",
        "category_share_prev",
        "log_rating_count",
        "log_min_installs",
        "log_age_days",
        "is_free",
        "is_mid_rank",
        "is_deep_rank",
        "has_rating_data",
        "has_install_data",
        "score_density",
    ]

    base_cols = feature_cols + [target, "transition_idx", "sample_weight"]
    model_df = transitions[base_cols].copy()

    # Focus model on hidden-gem candidate zone only.
    model_df = model_df[pd.to_numeric(model_df["rank"], errors="coerce") >= 11].copy()

    for col in ["is_free", "is_mid_rank", "is_deep_rank", "has_rating_data", "has_install_data"]:
        model_df[col] = pd.to_numeric(model_df[col], errors="coerce").fillna(0).astype(int)

    for col in ["source_confidence", "category_share_prev", "log_rating_count", "log_min_installs", "log_age_days", "score_density", "rank", "rank_inv", "rank_pct", "rank_sq"]:
        model_df[col] = pd.to_numeric(model_df[col], errors="coerce")

    model_df["sample_weight"] = pd.to_numeric(model_df["sample_weight"], errors="coerce").fillna(1.0)

    unique_steps = sorted(model_df["transition_idx"].dropna().unique())
    if len(unique_steps) >= 3:
        test_step = unique_steps[-1]
        train_df = model_df[model_df["transition_idx"] < test_step].copy()
        test_df = model_df[model_df["transition_idx"] == test_step].copy()
        if train_df.empty or test_df.empty or train_df[target].nunique() < 2:
            train_df, test_df = train_test_split(model_df, test_size=0.25, random_state=42, stratify=model_df[target])
    else:
        train_df, test_df = train_test_split(model_df, test_size=0.25, random_state=42, stratify=model_df[target])

    if train_df[target].nunique() < 2 or test_df[target].nunique() < 1:
        return None, {}, 0.5, train_df, test_df

    num_cols = [
        "rank", "rank_inv", "rank_pct", "rank_sq", "source_confidence", "category_share_prev",
        "log_rating_count", "log_min_installs", "log_age_days", "is_free", "is_mid_rank",
        "is_deep_rank", "has_rating_data", "has_install_data", "score_density",
    ]
    cat_cols = ["store", "category"]

    # Validation split for hyperparameter search.
    # Stratified random split is more stable than single-step validation for sparse hidden-gem labels.
    val_size = float(getattr(cfg, "model_validation_size", 0.2))
    val_size = min(0.4, max(0.1, val_size))
    tr_df, va_df = train_test_split(train_df, test_size=val_size, random_state=42, stratify=train_df[target])

    candidates = [
        {"n_estimators": 500, "max_depth": 14, "min_samples_leaf": 2, "min_samples_split": 2, "max_features": "sqrt"},
        {"n_estimators": 800, "max_depth": None, "min_samples_leaf": 1, "min_samples_split": 2, "max_features": 0.55},
        {"n_estimators": 700, "max_depth": 18, "min_samples_leaf": 1, "min_samples_split": 4, "max_features": 0.45},
        {"n_estimators": 350, "max_depth": 10, "min_samples_leaf": 3, "min_samples_split": 4, "max_features": "sqrt"},
        {"n_estimators": 900, "max_depth": 22, "min_samples_leaf": 2, "min_samples_split": 2, "max_features": 0.35},
    ]

    best = None
    for params in candidates:
        model = _build_pipeline(params, num_cols, cat_cols)
        model.fit(tr_df[feature_cols], tr_df[target], clf__sample_weight=tr_df["sample_weight"])

        va_proba = model.predict_proba(va_df[feature_cols])[:, 1]
        t = _best_threshold(va_df[target].to_numpy(), va_proba, min_precision=float(getattr(cfg, "model_threshold_min_precision", 0.62)))
        va_pred = (va_proba >= t).astype(int)

        p = precision_score(va_df[target], va_pred, zero_division=0)
        r = recall_score(va_df[target], va_pred, zero_division=0)
        f1 = f1_score(va_df[target], va_pred, zero_division=0)
        ap = average_precision_score(va_df[target], va_proba)
        score = 0.45 * ap + 0.40 * f1 + 0.15 * p

        if best is None or score > best["score"]:
            best = {
                "params": params,
                "threshold": t,
                "score": score,
                "val_precision": p,
                "val_recall": r,
                "val_f1": f1,
                "val_ap": ap,
            }

    if best is None:
        return None, {}, 0.5, train_df, test_df

    final_model = _build_pipeline(best["params"], num_cols, cat_cols)
    final_model.fit(train_df[feature_cols], train_df[target], clf__sample_weight=train_df["sample_weight"])

    test_proba = final_model.predict_proba(test_df[feature_cols])[:, 1]
    test_pred = (test_proba >= best["threshold"]).astype(int)

    metrics = {
        "accuracy": float(accuracy_score(test_df[target], test_pred)),
        "precision": float(precision_score(test_df[target], test_pred, zero_division=0)),
        "recall": float(recall_score(test_df[target], test_pred, zero_division=0)),
        "f1": float(f1_score(test_df[target], test_pred, zero_division=0)),
        "avg_precision": float(average_precision_score(test_df[target], test_proba)),
        "threshold": float(best["threshold"]),
        "val_precision": float(best["val_precision"]),
        "val_recall": float(best["val_recall"]),
        "val_f1": float(best["val_f1"]),
        "val_avg_precision": float(best["val_ap"]),
        "best_params": str(best["params"]),
    }
    try:
        metrics["roc_auc"] = float(roc_auc_score(test_df[target], test_proba))
    except Exception:
        metrics["roc_auc"] = float("nan")

    return final_model, metrics, float(best["threshold"]), train_df, test_df


def fit_stats_logit(transitions: pd.DataFrame):
    if transitions.empty or transitions["hidden_gem_30d"].nunique() < 2:
        return None, pd.DataFrame()

    d = transitions.copy()
    d = d[pd.to_numeric(d["rank"], errors="coerce") >= 11].copy()
    d["is_free"] = d.get("is_free", 0).fillna(0).astype(int)
    d["is_google"] = (d["store"] == "Google Play Store").astype(int)

    X = d[["rank", "rank_inv", "source_confidence", "log_rating_count", "log_min_installs", "log_age_days", "is_free", "is_google", "category_share_prev"]].copy()
    X = X.replace([np.inf, -np.inf], np.nan).fillna(0.0)
    y = d["hidden_gem_30d"].astype(int)
    w = pd.to_numeric(d.get("sample_weight"), errors="coerce").fillna(1.0)

    try:
        X = sm.add_constant(X, has_constant="add")
        m = sm.GLM(y, X, family=sm.families.Binomial(), freq_weights=w)
        r = m.fit()
    except Exception:
        return None, pd.DataFrame()

    coef = pd.DataFrame({
        "feature": r.params.index,
        "coef": r.params.values,
        "odds_ratio": np.exp(r.params.values),
        "p_value": r.pvalues.values,
    }).sort_values("p_value")

    return r, coef



In [ ]:
# Train ML + statistical model for 30-day hidden gems
transition_df = build_transition_dataset(config)
print(f"Transition rows: {len(transition_df)}")

ml_model = None
ml_metrics = {}
category_ml_forecast = pd.DataFrame()
app_ml_forecast = pd.DataFrame()
stats_result = None
stats_coef_df = pd.DataFrame()

if len(transition_df) < 120:
    print("Not enough transition data for robust hidden-gem modeling yet. Keep collecting snapshots.")
else:
    ml_model, ml_metrics, best_threshold, train_df, test_df = train_ml_forecast_model(transition_df, config)

    if ml_model is not None:
        print("ML hidden-gem metrics (30-day objective, tuned):")
        printable = {k: (round(v, 4) if isinstance(v, (int, float)) and v == v else v) for k, v in ml_metrics.items()}
        print(printable)

        pred_features = snapshot_df[[
            "rank", "store", "category", "source_confidence", "rating_count", "min_installs", "age_days", "is_free",
            "app_name", "app_id", "developer_name",
        ]].copy()

        pred_features["rank"] = pd.to_numeric(pred_features["rank"], errors="coerce")
        pred_features["source_confidence"] = pd.to_numeric(pred_features["source_confidence"], errors="coerce").fillna(0.5)
        pred_features["rating_count"] = pd.to_numeric(pred_features["rating_count"], errors="coerce")
        pred_features["min_installs"] = pd.to_numeric(pred_features["min_installs"], errors="coerce")
        pred_features["age_days"] = pd.to_numeric(pred_features["age_days"], errors="coerce")
        pred_features["is_free"] = pred_features["is_free"].fillna(False).astype(int)

        pred_features["rank_inv"] = (config.limit + 1) - pred_features["rank"]
        pred_features["rank_pct"] = pred_features["rank"] / max(config.limit, 1)
        pred_features["rank_sq"] = pred_features["rank"] ** 2
        pred_features["is_mid_rank"] = ((pred_features["rank"] >= config.hidden_gem_min_rank) & (pred_features["rank"] <= 30)).astype(int)
        pred_features["is_deep_rank"] = (pred_features["rank"] > 30).astype(int)

        pred_features["log_rating_count"] = np.log1p(pred_features["rating_count"].fillna(0))
        pred_features["log_min_installs"] = np.log1p(pred_features["min_installs"].fillna(0))
        pred_features["log_age_days"] = np.log1p(pred_features["age_days"].clip(lower=0).fillna(0))
        pred_features["has_rating_data"] = (pred_features["rating_count"].fillna(0) > 0).astype(int)
        pred_features["has_install_data"] = (pred_features["min_installs"].fillna(0) > 0).astype(int)
        pred_features["score_density"] = pred_features["source_confidence"] * pred_features["rank_inv"].clip(lower=0)

        # Approximate category prevalence in current snapshot for inference.
        grp = pred_features.groupby(["store", "category"]).size().rename("category_count").reset_index()
        st = pred_features.groupby(["store"]).size().rename("store_count").reset_index()
        pred_features = pred_features.merge(grp, on=["store", "category"], how="left").merge(st, on=["store"], how="left")
        pred_features["category_share_prev"] = pred_features["category_count"] / pred_features["store_count"].replace(0, np.nan)
        pred_features["category_share_prev"] = pred_features["category_share_prev"].fillna(0)

        X_pred = pred_features[[
            "rank", "rank_inv", "rank_pct", "rank_sq", "store", "category", "source_confidence", "category_share_prev",
            "log_rating_count", "log_min_installs", "log_age_days", "is_free", "is_mid_rank", "is_deep_rank",
            "has_rating_data", "has_install_data", "score_density",
        ]]
        pred_features["p_hidden_gem_30d"] = ml_model.predict_proba(X_pred)[:, 1]

        hidden_pool = pred_features[pred_features["rank"] >= config.hidden_gem_min_rank].copy()
        hidden_pool["pred_hidden_gem"] = (hidden_pool["p_hidden_gem_30d"] >= best_threshold).astype(int)

        app_ml_forecast = hidden_pool[[
            "store", "rank", "app_name", "category", "developer_name", "source_confidence", "p_hidden_gem_30d", "pred_hidden_gem"
        ]].sort_values(["p_hidden_gem_30d", "rank"], ascending=[False, True])

        category_ml_forecast = app_ml_forecast.groupby("category").agg(
            candidates=("app_name", "count"),
            avg_p_hidden_gem_30d=("p_hidden_gem_30d", "mean"),
            quality_weighted_p_hidden_gem_30d=("p_hidden_gem_30d", lambda s: float(np.average(s, weights=app_ml_forecast.loc[s.index, "source_confidence"].fillna(0.5)))),
            strong_candidates=("pred_hidden_gem", "sum"),
            best_rank=("rank", "min"),
        ).reset_index().sort_values(["quality_weighted_p_hidden_gem_30d", "strong_candidates"], ascending=[False, False])

        print("Top hidden-gem app candidates (30-day forecast):")
        display(app_ml_forecast.head(20))

        print("Top hidden-gem categories (30-day forecast):")
        display(category_ml_forecast.head(12))

    stats_result, stats_coef_df = fit_stats_logit(transition_df)
    if stats_result is not None and not stats_coef_df.empty:
        print("Statistical model (hidden_gem_30d) key coefficients:")
        display(stats_coef_df.head(12))
        pseudo_r2 = 1.0 - (stats_result.deviance / stats_result.null_deviance) if stats_result.null_deviance else float('nan')
        print(f"Logit pseudo-R2: {pseudo_r2:.4f}")



/tmp/ipython-input-335/1095962420.py:131: FutureWarning:

Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`

/tmp/ipython-input-335/1095962420.py:131: FutureWarning:

Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`

/tmp/ipython-input-335/1095962420.py:131: FutureWarning:

Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`

/tmp/ipython-input-335/1095962420.py:131: FutureWarning:

Downcasting object

Transition rows: 500
ML hidden-gem metrics (30-day objective, tuned):
{'accuracy': 1.0, 'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'avg_precision': 1.0, 'threshold': 0.72, 'val_precision': 0.6667, 'val_recall': 0.8, 'val_f1': 0.7273, 'val_avg_precision': 0.5292, 'best_params': "{'n_estimators': 700, 'max_depth': 18, 'min_samples_leaf': 1, 'min_samples_split': 4, 'max_features': 0.45}", 'roc_auc': 1.0}
Top hidden-gem app candidates (30-day forecast):


/tmp/ipython-input-335/1846867866.py:32: FutureWarning:

Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`



,store,rank,app_name,category,developer_name,source_confidence,p_hidden_gem_30d,pred_hidden_gem
83,Google Play Store,34,Messenger,Mixed / Proxy,"Meta Platforms, Inc.",0.6,0.730711,1
74,Google Play Store,25,LinkedIn: Community & Network,Mixed / Proxy,LinkedIn,0.6,0.727609,1
82,Google Play Store,33,"iHeart: Music, Radio, Podcasts",Mixed / Proxy,"iHeartMedia, Inc.",0.6,0.641797,0
80,Google Play Store,31,Google Meet,Mixed / Proxy,Google LLC,0.6,0.635486,0
75,Google Play Store,26,Gmail,Mixed / Proxy,Google LLC,0.6,0.634613,0
68,Google Play Store,19,Home Workout - No Equipment,Health & Fitness,Leap Fitness Group,0.6,0.537833,0
67,Google Play Store,18,Google,Tools,Google LLC,0.6,0.537080,0
81,Google Play Store,32,Threads,Mixed / Proxy,Instagram,0.6,0.446042,0
79,Google Play Store,30,Google Earth,Mixed / Proxy,Google LLC,0.6,0.439943,0
76,Google Play Store,27,Microsoft OneNote: Save Notes,Mixed / Proxy,Microsoft Corporation,0.6,0.434998,0


Top hidden-gem categories (30-day forecast):


,category,candidates,avg_p_hidden_gem_30d,quality_weighted_p_hidden_gem_30d,strong_candidates,best_rank
7,Health & Fitness,1,0.537833,0.537833,0,19
18,Tools,1,0.537080,0.537080,0,18
9,Mixed / Proxy,30,0.228019,0.228019,2,21
14,Photography,1,0.008153,0.008153,0,17
13,Photo & Video,4,0.000320,0.000320,0,21
4,Entertainment,12,0.000119,0.000077,0,14
0,Books & Reference,1,0.000000,0.000000,0,12
1,Business,1,0.000000,0.000000,0,48
2,Communication,1,0.000000,0.000000,0,13
3,Education,1,0.000000,0.000000,0,41


Statistical model (hidden_gem_30d) key coefficients:


,feature,coef,odds_ratio,p_value
1,rank,-0.338880,7.125681e-01,0.999865
0,const,-0.012893,9.871896e-01,0.999869
2,rank_inv,-0.318670,7.271155e-01,0.999873
3,source_confidence,-7.766607,4.236483e-04,0.999915
8,is_google,19.384284,2.621125e+08,0.999915
9,category_share_prev,0.517624,1.678035e+00,0.999998
4,log_rating_count,0.000000,1.000000e+00,NaN
5,log_min_installs,0.000000,1.000000e+00,NaN
6,log_age_days,0.000000,1.000000e+00,NaN
7,is_free,0.000000,1.000000e+00,NaN


Logit pseudo-R2: 0.2822


In [ ]:
# Strategy engine: convert market data into build + go-to-market recommendations
CATEGORY_PLAYBOOK = {
    "Productivity": {
        "app_type": "AI copilot for a specific professional workflow",
        "build_focus": "One core job-to-be-done, fast onboarding, reusable templates",
        "marketing": "Demo-led short videos, LinkedIn creators, SEO landing pages",
        "monetization": "Freemium + annual subscription",
    },
    "Social Networking": {
        "app_type": "Niche social/community app with creator loops",
        "build_focus": "Identity system, high-quality feed, creator tools",
        "marketing": "Creator seeding on TikTok/Instagram, ambassador program, referrals",
        "monetization": "Ads + subscriptions",
    },
    "Photo & Video": {
        "app_type": "Outcome-first editing app",
        "build_focus": "One-tap edits, trend templates, export quality",
        "marketing": "UGC challenges, creator bundles, watermark virality loop",
        "monetization": "Freemium + template pack IAP",
    },
    "Shopping": {
        "app_type": "Vertical shopping assistant",
        "build_focus": "Price alerts, trust signals, conversion-focused checkout",
        "marketing": "Affiliate creators, promo flywheel, seasonal campaigns",
        "monetization": "Affiliate + sponsored placements",
    },
    "Entertainment": {
        "app_type": "Habit-forming personalized content utility",
        "build_focus": "Recommendation quality and retention loops",
        "marketing": "Trend hijacking + creator collabs + referral loops",
        "monetization": "Ads + premium",
    },
}

CATEGORY_NORMALIZATION = {
    "Social": "Social Networking",
    "Music & Audio": "Music",
    "Books & Reference": "Reference",
}

DEFAULT_PLAYBOOK = {
    "app_type": "Niche utility solving one painful recurring problem",
    "build_focus": "Narrow MVP with one strong activation event",
    "marketing": "ASO + short demos + niche communities + creator partnerships",
    "monetization": "Freemium with paid power features",
}


def _meta_coverage(group: pd.DataFrame) -> float:
    cols = ["age_days", "rating_count", "min_installs"]
    cols = [c for c in cols if c in group.columns]
    if not cols:
        return 0.0
    return float(group[cols].notna().mean().mean())


def build_market_strategy(snapshot_df: pd.DataFrame, overlap_df: pd.DataFrame, momentum_df: pd.DataFrame | None, cfg: TrackerConfig):
    base = snapshot_df.copy()
    base["category"] = base["category"].fillna("Unknown")
    base["category"] = base["category"].replace(CATEGORY_NORMALIZATION)
    base = base[base["category"] != "Unknown"].copy()
    if base.empty:
        return pd.DataFrame(), pd.DataFrame(), "No category data available."

    cat = base.groupby("category").agg(
        app_count=("app_id", "count"),
        avg_rank=("rank", "mean"),
        top10=("rank", lambda s: int((pd.to_numeric(s, errors="coerce") <= 10).sum())),
        stores_covered=("store", "nunique"),
        new_app_count=("age_days", lambda s: int(((pd.to_numeric(s, errors="coerce") >= 0) & (pd.to_numeric(s, errors="coerce") <= cfg.new_app_days)).sum())),
        high_installs=("min_installs", lambda s: int((pd.to_numeric(s, errors="coerce") >= 1_000_000).sum())),
        strong_ratings=("rating_count", lambda s: int((pd.to_numeric(s, errors="coerce") >= 10000).sum())),
    ).reset_index()

    dev_counts = base.groupby(["category", "developer_name"]).size().reset_index(name="n")
    dev_share = dev_counts.groupby("category").apply(lambda g: float(g["n"].max() / g["n"].sum())).rename("max_dev_share").reset_index()
    cat = cat.merge(dev_share, on="category", how="left")

    if overlap_df is not None and not overlap_df.empty:
        apple_cat = base[base["store"] == "Apple App Store"][["app_id", "category"]].drop_duplicates()
        ov = overlap_df.merge(apple_cat, left_on="apple_app_id", right_on="app_id", how="left")
        ov_counts = ov.groupby("category").size().rename("overlap_matches").reset_index()
        cat = cat.merge(ov_counts, on="category", how="left")
    else:
        cat["overlap_matches"] = 0
    cat["overlap_matches"] = cat["overlap_matches"].fillna(0).astype(int)

    if momentum_df is not None and not momentum_df.empty:
        mom = momentum_df.groupby("category").agg(
            up_count=("status", lambda s: int((s == "up").sum())),
            down_count=("status", lambda s: int((s == "down").sum())),
            new_count=("status", lambda s: int((s == "new").sum())),
            dropped_count=("status", lambda s: int((s == "dropped").sum())),
        ).reset_index()
        cat = cat.merge(mom, on="category", how="left")
    else:
        cat["up_count"] = 0
        cat["down_count"] = 0
        cat["new_count"] = 0
        cat["dropped_count"] = 0

    for c in ["up_count", "down_count", "new_count", "dropped_count"]:
        cat[c] = cat[c].fillna(0).astype(int)

    meta_cov = base.groupby("category").apply(_meta_coverage).rename("meta_coverage").reset_index()
    cat = cat.merge(meta_cov, on="category", how="left")
    cat["meta_coverage"] = cat["meta_coverage"].fillna(0.0)

    total = max(1, int(cat["app_count"].sum()))
    max_apps = max(1, int(cat["app_count"].max()))
    max_top10 = max(1, int(cat["top10"].max()))
    max_overlap = max(1, int(cat["overlap_matches"].max()))

    cat["n_share"] = cat["app_count"] / total
    cat["n_rank"] = ((cfg.limit + 1) - cat["avg_rank"]) / cfg.limit
    cat["n_top10"] = cat["top10"] / max_top10
    cat["n_overlap"] = cat["overlap_matches"] / max_overlap
    cat["n_newness"] = cat["new_app_count"] / cat["app_count"].clip(lower=1)
    traction = (cat["high_installs"] + cat["strong_ratings"]) / cat["app_count"].clip(lower=1)
    cat["n_traction"] = traction / max(1e-6, float(traction.max()))

    growth = (
        cat["up_count"] + 0.5 * cat["new_count"] - cat["down_count"] - 0.25 * cat["dropped_count"]
    ) / cat["app_count"].clip(lower=1)
    cat["n_growth"] = ((growth.clip(-1, 1) + 1.0) / 2.0)

    cat["competition_score"] = 0.6 * (cat["app_count"] / max_apps) + 0.4 * cat["max_dev_share"].fillna(0)

    cat["market_score"] = 100 * (
        0.24 * cat["n_share"]
        + 0.20 * cat["n_rank"]
        + 0.14 * cat["n_top10"]
        + 0.14 * cat["n_growth"]
        + 0.10 * cat["n_newness"]
        + 0.10 * cat["n_overlap"]
        + 0.08 * cat["n_traction"]
        - 0.18 * cat["competition_score"]
    )

    cat["confidence_score"] = 100 * (
        0.45 * (cat["app_count"] / max_apps)
        + 0.30 * cat["meta_coverage"]
        + 0.25 * (cat["stores_covered"] / 2.0)
    )

    cat = cat.sort_values(["market_score", "app_count"], ascending=[False, False]).reset_index(drop=True)

    rows = []
    for r in cat.head(max(3, int(cfg.strategy_top_n))).itertuples(index=False):
        play = CATEGORY_PLAYBOOK.get(r.category, DEFAULT_PLAYBOOK)
        rows.append(
            {
                "category": r.category,
                "market_score": round(float(r.market_score), 1),
                "confidence_score": round(float(r.confidence_score), 1),
                "why_now": f"{int(r.app_count)} top apps, avg rank {r.avg_rank:.1f}, new={int(r.new_app_count)}, up={int(r.up_count)}, down={int(r.down_count)}",
                "recommended_app_type": play["app_type"],
                "build_focus": play["build_focus"],
                "marketing_plan": play["marketing"],
                "monetization": play["monetization"],
            }
        )
    plan_df = pd.DataFrame(rows)

    if plan_df.empty:
        brief = "No strategic recommendation available."
    else:
        best = plan_df.iloc[0]
        brief = "\n".join([
            "# Founder Brief",
            "",
            f"Top opportunity now: **{best['category']}**",
            f"Market score: **{best['market_score']}** | Confidence: **{best['confidence_score']}**",
            "",
            "## What to Build",
            f"- {best['recommended_app_type']}",
            f"- Build focus: {best['build_focus']}",
            "",
            "## How to Market",
            f"- {best['marketing_plan']}",
            "",
            "## Monetization",
            f"- {best['monetization']}",
            "",
            "## Why This Category Now",
            f"- {best['why_now']}",
        ])

    return cat, plan_df, brief




In [ ]:
# Market strategy output: what to build and how to market it
strategy_df, founder_plan_df, founder_brief_text = build_market_strategy(
    snapshot_df=snapshot_df,
    overlap_df=overlap_df,
    momentum_df=momentum_df if not momentum_df.empty else None,
    cfg=config,
)

if strategy_df.empty:
    print("Insufficient data for strategy output.")
else:
    if 'category_ml_forecast' in globals() and isinstance(category_ml_forecast, pd.DataFrame) and not category_ml_forecast.empty:
        strategy_df = strategy_df.merge(
            category_ml_forecast[["category", "avg_p_hidden_gem_30d", "strong_candidates"]],
            on="category",
            how="left",
        )
        med = strategy_df["avg_p_hidden_gem_30d"].median() if strategy_df["avg_p_hidden_gem_30d"].notna().any() else 0.5
        strategy_df["avg_p_hidden_gem_30d"] = strategy_df["avg_p_hidden_gem_30d"].fillna(med)
        strategy_df["strong_candidates"] = strategy_df["strong_candidates"].fillna(0)

        # Blend strategic score with hidden-gem ML signal.
        strategy_df["combined_score"] = (
            0.70 * strategy_df["market_score"]
            + 30.0 * strategy_df["avg_p_hidden_gem_30d"]
            + 1.0 * strategy_df["strong_candidates"]
        )
        strategy_df = strategy_df.sort_values(["combined_score", "market_score"], ascending=[False, False]).reset_index(drop=True)
    else:
        strategy_df["combined_score"] = strategy_df["market_score"]

    cols = [
        "category", "app_count", "avg_rank", "top10", "new_app_count",
        "up_count", "down_count", "overlap_matches", "competition_score",
        "market_score", "confidence_score", "combined_score",
    ]
    if "avg_p_hidden_gem_30d" in strategy_df.columns:
        cols += ["avg_p_hidden_gem_30d", "strong_candidates"]
    display(strategy_df[cols].head(int(config.strategy_top_n)))

    fig = px.scatter(
        strategy_df,
        x="competition_score",
        y="combined_score",
        size="app_count",
        color="category",
        hover_data=["confidence_score", "new_app_count", "up_count", "down_count"],
        title="Category Opportunity Map (30-day Hidden Gem Weighted)",
    )
    fig.show()

if not founder_plan_df.empty:
    display(founder_plan_df.head(5))

if 'avg_p_hidden_gem_30d' in strategy_df.columns:
    top_ml = strategy_df.iloc[0]
    founder_brief_text += (
        f"\n\n## Model Overlay (30-day Hidden Gem)"
        f"\n- Predicted hidden-gem probability for this category: {top_ml['avg_p_hidden_gem_30d']:.2f}"
        f"\n- Strong hidden-gem candidates in category: {int(top_ml.get('strong_candidates', 0))}"
    )

if 'app_ml_forecast' in globals() and isinstance(app_ml_forecast, pd.DataFrame) and not app_ml_forecast.empty:
    top_apps = app_ml_forecast.head(5)[["app_name", "category", "p_hidden_gem_30d"]]
    founder_brief_text += "\n\n## Top Hidden-Gem App Candidates\n"
    for r in top_apps.itertuples(index=False):
        founder_brief_text += f"- {r.app_name} ({r.category}) p={r.p_hidden_gem_30d:.2f}\n"

print("\n" + founder_brief_text)

age = pd.to_numeric(snapshot_df.get("age_days"), errors="coerce")
recent_top20 = snapshot_df[(age >= 0) & (age <= config.new_app_days) & (snapshot_df["rank"] <= 20)]
print(f"\nNew app surge signal (<= {config.new_app_days} days old and rank<=20): {len(recent_top20)} apps")
if not recent_top20.empty:
    display(recent_top20[["store", "rank", "app_name", "category", "age_days"]].sort_values(["store", "rank"]).head(20))



/tmp/ipython-input-335/2393541409.py:76: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.

/tmp/ipython-input-335/2393541409.py:105: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



,category,app_count,avg_rank,top10,new_app_count,up_count,down_count,overlap_matches,competition_score,market_score,confidence_score,combined_score,avg_p_hidden_gem_30d,strong_candidates
0,Social Networking,9,16.000000,4,0,0,0,3,0.224444,48.453333,58.5,33.917333,0.000000,0.0
1,Tools,1,18.000000,0,0,0,0,0,0.420000,20.880000,34.0,30.728387,0.537080,0.0
2,Health & Fitness,1,19.000000,0,0,0,0,0,0.420000,20.480000,34.0,30.470982,0.537833,0.0
3,Productivity,8,8.250000,4,0,0,0,0,0.260000,40.840000,57.0,28.588000,0.000000,0.0
4,Reference,3,9.333333,2,0,0,0,0,0.193333,35.906667,37.0,25.134667,0.000000,0.0
5,Music,3,11.666667,1,0,0,0,1,0.193333,33.473333,49.5,23.431333,0.000000,0.0
6,Communication,2,7.000000,1,0,0,0,0,0.240000,32.260000,35.5,22.582000,0.000000,0.0
7,Photo & Video,5,33.200000,1,0,0,0,3,0.180000,29.580000,40.0,20.715593,0.000320,0.0


,category,market_score,confidence_score,why_now,recommended_app_type,build_focus,marketing_plan,monetization
0,Social Networking,48.5,58.5,"9 top apps, avg rank 16.0, new=0, up=0, down=0",Niche social/community app with creator loops,"Identity system, high-quality feed, creator tools","Creator seeding on TikTok/Instagram, ambassado...",Ads + subscriptions
1,Productivity,40.8,57.0,"8 top apps, avg rank 8.2, new=0, up=0, down=0",AI copilot for a specific professional workflow,"One core job-to-be-done, fast onboarding, reus...","Demo-led short videos, LinkedIn creators, SEO ...",Freemium + annual subscription
2,Reference,35.9,37.0,"3 top apps, avg rank 9.3, new=0, up=0, down=0",Niche utility solving one painful recurring pr...,Narrow MVP with one strong activation event,ASO + short demos + niche communities + creato...,Freemium with paid power features
3,Music,33.5,49.5,"3 top apps, avg rank 11.7, new=0, up=0, down=0",Niche utility solving one painful recurring pr...,Narrow MVP with one strong activation event,ASO + short demos + niche communities + creato...,Freemium with paid power features
4,Communication,32.3,35.5,"2 top apps, avg rank 7.0, new=0, up=0, down=0",Niche utility solving one painful recurring pr...,Narrow MVP with one strong activation event,ASO + short demos + niche communities + creato...,Freemium with paid power features



# Founder Brief

Top opportunity now: **Social Networking**
Market score: **48.5** | Confidence: **58.5**

## What to Build
- Niche social/community app with creator loops
- Build focus: Identity system, high-quality feed, creator tools

## How to Market
- Creator seeding on TikTok/Instagram, ambassador program, referrals

## Monetization
- Ads + subscriptions

## Why This Category Now
- 9 top apps, avg rank 16.0, new=0, up=0, down=0

## Model Overlay (30-day Hidden Gem)
- Predicted hidden-gem probability for this category: 0.00
- Strong hidden-gem candidates in category: 0

## Top Hidden-Gem App Candidates
- Messenger (Mixed / Proxy) p=0.73
- LinkedIn: Community & Network (Mixed / Proxy) p=0.73
- iHeart: Music, Radio, Podcasts (Mixed / Proxy) p=0.64
- Google Meet (Mixed / Proxy) p=0.64
- Gmail (Mixed / Proxy) p=0.63


New app surge signal (<= 180 days old and rank<=20): 0 apps


In [ ]:
# HTML report template (Jinja2)
MARKET_REPORT_TEMPLATE = r"""
<!doctype html>
<html lang="en">
<head>
  <meta charset="utf-8" />
  <meta name="viewport" content="width=device-width, initial-scale=1" />
  <title>Market Analysis Report - {{ snapshot_id }}</title>
  <style>
    :root {
      --bg: #f4f6f8;
      --card: #ffffff;
      --ink: #102a43;
      --muted: #52606d;
      --line: #d9e2ec;
      --brand: #1f6feb;
      --accent: #0f766e;
    }
    * { box-sizing: border-box; }
    body {
      margin: 0;
      font-family: "Segoe UI", "Helvetica Neue", Arial, sans-serif;
      color: var(--ink);
      background: linear-gradient(180deg, #eef4fb 0%, var(--bg) 38%, #ecfdf5 100%);
      line-height: 1.5;
    }
    .container { max-width: 1240px; margin: 0 auto; padding: 24px; }
    .hero {
      background: linear-gradient(135deg, #0b3d91 0%, #1f6feb 55%, #0f766e 100%);
      color: #fff;
      border-radius: 16px;
      padding: 24px;
      box-shadow: 0 12px 28px rgba(15, 23, 42, 0.20);
    }
    .hero h1 { margin: 0; font-size: 28px; }
    .hero p { margin: 6px 0 0; color: rgba(255,255,255,0.92); }
    .metrics-grid {
      margin-top: 16px;
      display: grid;
      gap: 12px;
      grid-template-columns: repeat(auto-fit, minmax(170px, 1fr));
    }
    .metric-card {
      background: rgba(255,255,255,0.14);
      border: 1px solid rgba(255,255,255,0.22);
      border-radius: 12px;
      padding: 10px 12px;
      backdrop-filter: blur(2px);
    }
    .metric-label { font-size: 12px; text-transform: uppercase; letter-spacing: 0.03em; opacity: 0.92; }
    .metric-value { margin-top: 4px; font-size: 19px; font-weight: 700; }
    .card {
      margin-top: 16px;
      background: var(--card);
      border: 1px solid var(--line);
      border-radius: 14px;
      padding: 14px;
      box-shadow: 0 7px 18px rgba(15, 23, 42, 0.06);
    }
    h2 { margin: 8px 0 0; font-size: 20px; color: #16324f; }
    h3 { margin: 0 0 10px; font-size: 16px; color: #1f3b57; }
    .table { width: 100%; border-collapse: collapse; }
    .table th, .table td { border-bottom: 1px solid var(--line); padding: 8px; text-align: left; font-size: 13px; vertical-align: top; }
    .table th { background: #f8fbff; color: #102a43; }
    .table tr:hover td { background: #f7fbff; }
    .muted { color: var(--muted); margin: 0; }
    .brief { white-space: pre-wrap; background: #f8fbff; border: 1px solid var(--line); border-radius: 10px; padding: 12px; font-size: 13px; }
    .grid-2 { display: grid; grid-template-columns: 1fr 1fr; gap: 16px; }
    @media (max-width: 900px) { .grid-2 { grid-template-columns: 1fr; } }
  </style>
</head>
<body>
  <div class="container">
    <section class="hero">
      <h1>App Market Analysis Report</h1>
      <p>Snapshot <strong>{{ snapshot_id }}</strong> | Country: <strong>{{ country }}</strong></p>
      <div class="metrics-grid">
        {% for item in summary_cards %}
        <div class="metric-card">
          <div class="metric-label">{{ item.label }}</div>
          <div class="metric-value">{{ item.value }}</div>
        </div>
        {% endfor %}
      </div>
    </section>

    <section class="grid-2">
      <article class="card">
        <h2>Strategy Shortlist</h2>
        {{ strategy_table_html | safe }}
      </article>
      <article class="card">
        <h2>ML Metrics</h2>
        {{ ml_metrics_table_html | safe }}
      </article>
    </section>

    <section class="grid-2">
      <article class="card">
        <h2>Founder Action Plan</h2>
        {{ founder_plan_table_html | safe }}
      </article>
      <article class="card">
        <h2>Top Hidden-Gem Apps</h2>
        {{ top_apps_table_html | safe }}
      </article>
    </section>

    <section class="card">
      <h2>Founder Brief</h2>
      <div class="brief">{{ founder_brief_text }}</div>
    </section>

    <section class="card">
      <h2>All Plots</h2>
      {% if figure_blocks %}
        {% for fig in figure_blocks %}
        <section class="card">
          <h3>{{ fig.title }}</h3>
          {{ fig.html | safe }}
        </section>
        {% endfor %}
      {% else %}
        <p class="muted">No plots were generated for this run.</p>
      {% endif %}
    </section>
  </div>
</body>
</html>

"""


In [ ]:
# Save report artifacts (including strategy + model outputs + HTML dashboard)
from jinja2 import Environment, select_autoescape

reports_dir = config.storage_dir / "reports" / f"snapshot_{snapshot_df['snapshot_id'].iloc[0]}"
artifacts = build_reports(snapshot_df, overlap_df, momentum_df if not momentum_df.empty else None, reports_dir)

if 'strategy_df' in globals() and isinstance(strategy_df, pd.DataFrame) and not strategy_df.empty:
    strategy_path = reports_dir / "category_strategy.csv"
    strategy_df.to_csv(strategy_path, index=False)
    artifacts["category_strategy"] = strategy_path

if 'founder_plan_df' in globals() and isinstance(founder_plan_df, pd.DataFrame) and not founder_plan_df.empty:
    founder_plan_path = reports_dir / "founder_plan.csv"
    founder_plan_df.to_csv(founder_plan_path, index=False)
    artifacts["founder_plan"] = founder_plan_path

if 'founder_brief_text' in globals() and isinstance(founder_brief_text, str) and founder_brief_text.strip():
    founder_brief_path = reports_dir / "founder_brief.md"
    founder_brief_path.write_text(founder_brief_text, encoding="utf-8")
    artifacts["founder_brief"] = founder_brief_path

if 'app_ml_forecast' in globals() and isinstance(app_ml_forecast, pd.DataFrame) and not app_ml_forecast.empty:
    app_ml_path = reports_dir / "ml_hidden_gem_app_forecast_30d.csv"
    app_ml_forecast.to_csv(app_ml_path, index=False)
    artifacts["ml_hidden_gem_app_forecast_30d"] = app_ml_path

if 'category_ml_forecast' in globals() and isinstance(category_ml_forecast, pd.DataFrame) and not category_ml_forecast.empty:
    cat_ml_path = reports_dir / "ml_hidden_gem_category_forecast_30d.csv"
    category_ml_forecast.to_csv(cat_ml_path, index=False)
    artifacts["ml_hidden_gem_category_forecast_30d"] = cat_ml_path

if 'stats_coef_df' in globals() and isinstance(stats_coef_df, pd.DataFrame) and not stats_coef_df.empty:
    stats_path = reports_dir / "stats_hidden_gem_model_coefficients.csv"
    stats_coef_df.to_csv(stats_path, index=False)
    artifacts["stats_hidden_gem_model_coefficients"] = stats_path


def _safe_table(df: pd.DataFrame | None, cols: list[str] | None = None, n: int = 12) -> str:
    if df is None or not isinstance(df, pd.DataFrame) or df.empty:
        return '<p class="muted">No data available for this section.</p>'

    out = df.copy()
    if cols:
        keep = [c for c in cols if c in out.columns]
        if keep:
            out = out[keep]
    out = out.head(n)

    for col in out.columns:
        if pd.api.types.is_float_dtype(out[col]):
            out[col] = out[col].map(lambda v: "" if pd.isna(v) else f"{float(v):.4f}")

    return out.to_html(index=False, classes="table", border=0, justify="left")


def _metric_table(metrics: dict[str, Any] | None) -> str:
    if not metrics:
        return '<p class="muted">ML metrics are not available for this run.</p>'

    metric_rows = []
    for k, v in metrics.items():
        if isinstance(v, float):
            metric_rows.append({"metric": k, "value": f"{v:.4f}"})
        else:
            metric_rows.append({"metric": k, "value": str(v)})
    return pd.DataFrame(metric_rows).to_html(index=False, classes="table", border=0, justify="left")


def _make_report_figures() -> list[dict[str, str]]:
    blocks: list[dict[str, str]] = []
    include_plotly = "cdn"

    def add_fig(title: str, fig_obj: Any) -> None:
        nonlocal include_plotly
        fig_html = fig_obj.to_html(full_html=False, include_plotlyjs=include_plotly, config={"displayModeBar": False, "responsive": True})
        include_plotly = False
        blocks.append({"title": title, "html": fig_html})

    store_mix = snapshot_df["store"].value_counts().rename_axis("store").reset_index(name="count")
    if not store_mix.empty:
        f_store = px.bar(store_mix, x="store", y="count", color="store", title="Store Distribution", text_auto=True)
        f_store.update_layout(showlegend=False)
        add_fig("Store Distribution", f_store)

    apple_categories = (
        snapshot_df[snapshot_df["store"].eq("Apple App Store")]["category"]
        .fillna("Unknown")
        .value_counts()
        .rename_axis("category")
        .reset_index(name="count")
        .head(15)
    )
    if not apple_categories.empty:
        f_apple = px.bar(
            apple_categories,
            x="category",
            y="count",
            color="count",
            color_continuous_scale="Blues",
            title="Top Apple Categories",
        )
        add_fig("Top Apple Categories", f_apple)

    if isinstance(overlap_df, pd.DataFrame) and not overlap_df.empty and "match_score" in overlap_df.columns:
        f_overlap = px.histogram(overlap_df, x="match_score", nbins=14, title="Cross-Platform Match Score Distribution")
        add_fig("Overlap Match Scores", f_overlap)

    if isinstance(momentum_df, pd.DataFrame) and not momentum_df.empty and "status" in momentum_df.columns:
        momentum_status = momentum_df["status"].value_counts().rename_axis("status").reset_index(name="count")
        f_momentum = px.bar(momentum_status, x="status", y="count", color="status", title="Momentum Status Counts", text_auto=True)
        add_fig("Momentum Status", f_momentum)

    if 'app_ml_forecast' in globals() and isinstance(app_ml_forecast, pd.DataFrame) and not app_ml_forecast.empty:
        top_apps = app_ml_forecast.sort_values(["p_hidden_gem_30d", "rank"], ascending=[False, True]).head(20).copy().iloc[::-1]
        f_hidden_apps = px.bar(
            top_apps,
            x="p_hidden_gem_30d",
            y="app_name",
            color="store",
            orientation="h",
            hover_data=["rank", "category", "developer_name"],
            title="Top Hidden-Gem App Probabilities (30d)",
        )
        add_fig("Hidden-Gem App Probabilities", f_hidden_apps)

    if 'category_ml_forecast' in globals() and isinstance(category_ml_forecast, pd.DataFrame) and not category_ml_forecast.empty:
        top_cat_ml = category_ml_forecast.sort_values("quality_weighted_p_hidden_gem_30d", ascending=False).head(12)
        f_hidden_cat = px.bar(
            top_cat_ml,
            x="category",
            y="quality_weighted_p_hidden_gem_30d",
            color="strong_candidates",
            title="Category Hidden-Gem Potential (30d)",
            hover_data=["candidates", "best_rank"],
        )
        add_fig("Hidden-Gem Category Potential", f_hidden_cat)

    if 'strategy_df' in globals() and isinstance(strategy_df, pd.DataFrame) and not strategy_df.empty:
        y_axis = "combined_score" if "combined_score" in strategy_df.columns else "market_score"
        f_strategy = px.scatter(
            strategy_df,
            x="competition_score",
            y=y_axis,
            size="app_count",
            color="category",
            hover_data=["confidence_score", "new_app_count", "up_count", "down_count"],
            title="Category Opportunity Map",
        )
        add_fig("Category Opportunity Map", f_strategy)

    return blocks


def _build_summary_cards() -> list[dict[str, str]]:
    cards = [
        {"label": "Snapshot", "value": snapshot_id_value},
        {"label": "Captured UTC", "value": captured_value},
        {"label": "Country", "value": country_value.upper()},
        {"label": "Apps Analyzed", "value": f"{len(snapshot_df):,}"},
        {"label": "Cross-Platform Matches", "value": f"{len(overlap_df):,}" if isinstance(overlap_df, pd.DataFrame) else "0"},
        {"label": "Momentum Rows", "value": f"{len(momentum_df):,}" if isinstance(momentum_df, pd.DataFrame) else "0"},
    ]
    if 'app_ml_forecast' in globals() and isinstance(app_ml_forecast, pd.DataFrame) and not app_ml_forecast.empty and "pred_hidden_gem" in app_ml_forecast.columns:
        cards.append({"label": "Predicted Hidden Gems", "value": f"{int(app_ml_forecast['pred_hidden_gem'].sum()):,}"})
    return cards


snapshot_id_value = str(snapshot_df["snapshot_id"].iloc[0]) if "snapshot_id" in snapshot_df.columns and not snapshot_df.empty else "unknown"
captured_value = str(snapshot_df["captured_at_utc"].iloc[0]) if "captured_at_utc" in snapshot_df.columns and not snapshot_df.empty else "unknown"
country_value = str(snapshot_df["country"].iloc[0]) if "country" in snapshot_df.columns and not snapshot_df.empty else config.country

strategy_table_html = _safe_table(
    strategy_df if 'strategy_df' in globals() else None,
    ["category", "combined_score", "market_score", "competition_score", "app_count", "new_app_count", "up_count", "down_count"],
    n=int(getattr(config, "strategy_top_n", 8)),
)
ml_metrics_table_html = _metric_table(ml_metrics if 'ml_metrics' in globals() else None)
founder_plan_table_html = _safe_table(
    founder_plan_df if 'founder_plan_df' in globals() else None,
    ["category", "app_type", "build_focus", "marketing", "monetization", "why_now"],
    n=8,
)
top_apps_table_html = _safe_table(
    app_ml_forecast if 'app_ml_forecast' in globals() else None,
    ["store", "rank", "app_name", "category", "p_hidden_gem_30d", "pred_hidden_gem"],
    n=20,
)

summary_cards = _build_summary_cards()
figure_blocks = _make_report_figures()

template_text = globals().get("MARKET_REPORT_TEMPLATE")
if not template_text or not str(template_text).strip():
    raise ValueError("MARKET_REPORT_TEMPLATE is empty. Run the template cell before report export.")

env = Environment(
    autoescape=select_autoescape(["html", "xml"]),
)
template = env.from_string(str(template_text))

report_html = template.render(
    snapshot_id=snapshot_id_value,
    country=country_value.upper(),
    summary_cards=summary_cards,
    strategy_table_html=strategy_table_html,
    ml_metrics_table_html=ml_metrics_table_html,
    founder_plan_table_html=founder_plan_table_html,
    top_apps_table_html=top_apps_table_html,
    founder_brief_text=founder_brief_text if 'founder_brief_text' in globals() else "No founder brief generated in this run.",
    figure_blocks=figure_blocks,
)

html_report_path = reports_dir / "market_analysis_report.html"
html_report_path.write_text(report_html, encoding="utf-8")
artifacts["market_analysis_report"] = html_report_path

for name, path in artifacts.items():
    print(f"{name}: {path}")


store_distribution: data/reports/snapshot_20260302T133414Z/store_distribution.csv
overlap_summary: data/reports/snapshot_20260302T133414Z/overlap_summary.csv
momentum_summary: data/reports/snapshot_20260302T133414Z/momentum_summary.csv
category_strategy: data/reports/snapshot_20260302T133414Z/category_strategy.csv
founder_plan: data/reports/snapshot_20260302T133414Z/founder_plan.csv
founder_brief: data/reports/snapshot_20260302T133414Z/founder_brief.md
ml_hidden_gem_app_forecast_30d: data/reports/snapshot_20260302T133414Z/ml_hidden_gem_app_forecast_30d.csv
ml_hidden_gem_category_forecast_30d: data/reports/snapshot_20260302T133414Z/ml_hidden_gem_category_forecast_30d.csv
stats_hidden_gem_model_coefficients: data/reports/snapshot_20260302T133414Z/stats_hidden_gem_model_coefficients.csv
market_analysis_report: data/reports/snapshot_20260302T133414Z/market_analysis_report.html


## Next Steps

- Run daily for trend stability and use weekly averages of `market_score` before committing roadmap.
- Compare `us`, `gb`, `ca` and pick categories that stay top-3 in at least two markets.
- Use `founder_plan.csv` + `founder_brief.md` as your weekly product/marketing decision memo.
- Launch one narrow MVP first, measure CAC/retention, then expand scope.

